In [ ]:
import os
import io
import sys
import subprocess
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OpenAI API key loaded.")
else:
    print("OPENAI_API_KEY is missing. Add it to your .env file.")

client = OpenAI()

In [ ]:
PROJECT_DIR = Path.cwd()
GENERATED_DIR = PROJECT_DIR / "generated"
GENERATED_DIR.mkdir(exist_ok=True)

TARGET_LANGUAGE = "C++"
TARGET_EXTENSION = "cpp"

# مهم لجهازك لأن g++ موجود داخل MSYS2
os.environ["PATH"] = r"C:\msys64\ucrt64\bin;" + os.environ["PATH"]

compile_command = [
    "g++",
    "-std=c++20",
    "-O3",
    str(GENERATED_DIR / "main.cpp"),
    "-o",
    str(GENERATED_DIR / "main.exe"),
]

run_command = [str(GENERATED_DIR / "main.exe")]

MODEL = "gpt-5"  # أرخص من GPT-5 كبداية. تقدر تغيره إلى gpt-5 لاحقًا.

print("Project:", PROJECT_DIR)
print("Generated:", GENERATED_DIR)
print("Target:", TARGET_LANGUAGE)

In [ ]:
sample_python_code = """
def factorial(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

print("Factorial:", factorial(10))
"""

In [ ]:
def run_python(code: str) -> str:
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Python Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output


print(run_python(sample_python_code))

In [ ]:
system_prompt = f"""
You are an expert software engineer.
Your task is to convert Python code into high-performance {TARGET_LANGUAGE} code.

Rules:
- Return only valid {TARGET_LANGUAGE} code.
- Do not include markdown fences.
- Do not explain anything outside the code.
- The generated code must compile and run.
- The output must match the Python program output as closely as possible.
- Prefer simple, correct code first, then optimize when safe.
"""

def user_prompt_for_porting(python_code: str) -> str:
    return f"""
Convert this Python code into {TARGET_LANGUAGE}.

The code will be saved as generated/main.{TARGET_EXTENSION}.
It will be compiled using this command:
{compile_command}

It will be run using:
{run_command}

Return only {TARGET_LANGUAGE} code.

Python code:
{python_code}
"""

In [ ]:
def clean_model_code(text: str) -> str:
    return (
        text.replace("```cpp", "")
            .replace("```c++", "")
            .replace("```rust", "")
            .replace("```python", "")
            .replace("```", "")
            .strip()
    )


def port_code(python_code: str, model: str = MODEL) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt_for_porting(python_code)}
        ],
    )

    code = response.choices[0].message.content
    return clean_model_code(code)


cpp_code = port_code(sample_python_code)
print(cpp_code[:1000])

In [ ]:
def write_generated_code(code: str) -> Path:
    output_path = GENERATED_DIR / f"main.{TARGET_EXTENSION}"
    output_path.write_text(code, encoding="utf-8")
    return output_path


path = write_generated_code(cpp_code)
print("Wrote:", path)

In [ ]:
def compile_generated_code() -> tuple[bool, str]:
    result = subprocess.run(
        compile_command,
        text=True,
        capture_output=True
    )

    if result.returncode != 0:
        error = result.stderr or result.stdout or "Compilation failed with no output."
        return False, error

    return True, "Compilation succeeded."


def run_generated_program() -> tuple[bool, str]:
    result = subprocess.run(
        run_command,
        text=True,
        capture_output=True
    )

    if result.returncode != 0:
        error = result.stderr or result.stdout or "Program failed with no output."
        return False, error

    return True, result.stdout


ok, compile_output = compile_generated_code()
print(compile_output)

if ok:
    ok_run, run_output = run_generated_program()
    print(run_output)

In [ ]:
def user_prompt_for_fixing(
    original_python: str,
    broken_code: str,
    compiler_error: str
) -> str:
    return f"""
The generated {TARGET_LANGUAGE} code failed to compile or run.

Original Python code:
{original_python}

Broken {TARGET_LANGUAGE} code:
{broken_code}

Compiler/runtime error:
{compiler_error}

Fix the {TARGET_LANGUAGE} code.
Rules:
- Return only corrected {TARGET_LANGUAGE} code.
- Do not include markdown fences.
- Keep the output matching the Python code.
- Make the code compile and run on Windows with this compile command:
{compile_command}
"""


def fix_code(
    original_python: str,
    broken_code: str,
    compiler_error: str,
    model: str = MODEL
) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": user_prompt_for_fixing(
                    original_python,
                    broken_code,
                    compiler_error
                )
            }
        ],
    )

    fixed = response.choices[0].message.content
    return clean_model_code(fixed)

In [ ]:
def codeport_agent(
    python_code: str,
    model: str = MODEL,
    max_attempts: int = 3
) -> dict:
    logs = []
    python_output = run_python(python_code)

    logs.append("=== Python Output ===")
    logs.append(python_output)

    generated_code = port_code(python_code, model=model)

    for attempt in range(1, max_attempts + 1):
        logs.append(f"\n=== Attempt {attempt} ===")

        write_generated_code(generated_code)

        compile_ok, compile_msg = compile_generated_code()
        logs.append("Compile result:")
        logs.append(compile_msg)

        if not compile_ok:
            if attempt == max_attempts:
                return {
                    "success": False,
                    "python_output": python_output,
                    "generated_code": generated_code,
                    "program_output": "",
                    "logs": "\n".join(logs),
                }

            logs.append("Asking model to fix compile error...")
            generated_code = fix_code(
                original_python=python_code,
                broken_code=generated_code,
                compiler_error=compile_msg,
                model=model
            )
            continue

        run_ok, run_msg = run_generated_program()
        logs.append("Run result:")
        logs.append(run_msg)

        if run_ok:
            return {
                "success": True,
                "python_output": python_output,
                "generated_code": generated_code,
                "program_output": run_msg,
                "logs": "\n".join(logs),
            }

        if attempt == max_attempts:
            return {
                "success": False,
                "python_output": python_output,
                "generated_code": generated_code,
                "program_output": run_msg,
                "logs": "\n".join(logs),
            }

        logs.append("Asking model to fix runtime error...")
        generated_code = fix_code(
            original_python=python_code,
            broken_code=generated_code,
            compiler_error=run_msg,
            model=model
        )

    return {
        "success": False,
        "python_output": python_output,
        "generated_code": generated_code,
        "program_output": "",
        "logs": "\n".join(logs),
    }


result = codeport_agent(sample_python_code, model=MODEL, max_attempts=3)

print("Success:", result["success"])
print(result["logs"])

In [ ]:
harder_python_code = """
def max_subarray_sum(nums):
    best = nums[0]
    current = nums[0]

    for x in nums[1:]:
        current = max(x, current + x)
        best = max(best, current)

    return best

nums = [3, -5, 10, -2, 4, -20, 7, 8]
print("Max subarray sum:", max_subarray_sum(nums))
"""

result = codeport_agent(harder_python_code, model=MODEL, max_attempts=3)

print("Success:", result["success"])
print("Python output:")
print(result["python_output"])
print("Generated output:")
print(result["program_output"])
print("Logs:")
print(result["logs"])

In [ ]:
def gradio_agent(python_code: str, model: str, max_attempts: int):
    result = codeport_agent(
        python_code=python_code,
        model=model,
        max_attempts=max_attempts
    )

    return (
        result["generated_code"],
        result["python_output"],
        result["program_output"],
        result["logs"]
    )


available_models = [
    "gpt-5-mini",
    "gpt-5",
]

with gr.Blocks(title="CodePort Agent") as demo:
    gr.Markdown("# CodePort Agent")
    gr.Markdown("Convert Python code to C++ with compile/run and auto-fix loop.")

    with gr.Row():
        python_input = gr.Code(
            label="Python code",
            value=sample_python_code,
            language="python",
            lines=18
        )

        cpp_output = gr.Code(
            label="Generated C++ code",
            language="cpp",
            lines=18
        )

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=available_models,
            value="gpt-5-mini",
            label="Model"
        )

        attempts_slider = gr.Slider(
            minimum=1,
            maximum=5,
            value=3,
            step=1,
            label="Max repair attempts"
        )

        run_button = gr.Button("Run CodePort Agent")

    with gr.Row():
        python_result = gr.TextArea(label="Python output", lines=6)
        generated_result = gr.TextArea(label="Generated program output", lines=6)

    logs_output = gr.TextArea(label="Agent logs", lines=14)

    run_button.click(
        fn=gradio_agent,
        inputs=[python_input, model_dropdown, attempts_slider],
        outputs=[cpp_output, python_result, generated_result, logs_output]
    )

demo.launch(inbrowser=True)